In [0]:
# Portfolio Project: E-Commerce Sales Pipeline
# Layer: Data Quality — Checks & Monitoring

# Data Quality Checks

This pipeline running two apporaches of data quality:

| Approach | Usage |
|---|---|
| **Built-in PySpark checks** | light validation, fast, without library installation |
| **Great Expectations** | full validation, automatic documentation |

Validation results saved to Delta table `dq_results` for historical monitoring.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from datetime import datetime
import json

## Part 1: Built-in PySpark Data Quality Checks
No Installation needed.

In [0]:
class DataQualityChecker:
    """
    class to run data quality checks to Spark DataFrame.
    Store result to Delta table for historical monitoring.
    """

    def __init__(self, df: DataFrame, table_name: str, layer: str):
        self.df         = df
        self.table_name = table_name
        self.layer      = layer
        self.results    = []
        self.run_time   = datetime.now().isoformat()
        self.total_rows = df.count()

    def _add_result(self, check_name, passed, value, threshold=None, details=""):
        self.results.append({
            "run_time":   self.run_time,
            "table":      self.table_name,
            "layer":      self.layer,
            "check":      check_name,
            "passed":     passed,
            "value":      str(value),
            "threshold":  str(threshold) if threshold else "",
            "details":    details,
        })
        status = "✅ PASS" if passed else "❌ FAIL"
        print(f"  {status}  {check_name}: {value}"
              + (f" (threshold: {threshold})" if threshold else ""))
        return passed

    # ── Core checks ──────────────────────────────────────────

    def check_row_count(self, min_rows: int):
        """to ensure the table is not empty atau too small."""
        passed = self.total_rows >= min_rows
        return self._add_result(
            "row_count",
            passed,
            f"{self.total_rows:,} rows",
            f">= {min_rows:,}",
        )

    def check_no_nulls(self, columns: list):
        """Check no NULL value in the critical columns."""
        for col in columns:
            null_count = self.df.filter(F.col(col).isNull()).count()
            pct        = round(null_count / self.total_rows * 100, 2)
            passed     = null_count == 0
            self._add_result(
                f"no_nulls:{col}",
                passed,
                f"{null_count:,} nulls ({pct}%)",
                "0 nulls",
            )

    def check_uniqueness(self, columns: list):
        """Check a must-unique column (primary key)."""
        for col in columns:
            distinct = self.df.select(col).distinct().count()
            dupes    = self.total_rows - distinct
            passed   = dupes == 0
            self._add_result(
                f"uniqueness:{col}",
                passed,
                f"{dupes:,} duplicates",
                "0 duplicates",
            )

    def check_value_range(self, column: str, min_val=None, max_val=None):
        """Check numerical value in a fair range."""
        condition = F.lit(False)
        if min_val is not None:
            condition = condition | (F.col(column) < min_val)
        if max_val is not None:
            condition = condition | (F.col(column) > max_val)
        violations = self.df.filter(condition).count()
        passed     = violations == 0
        range_str  = f"[{min_val}, {max_val}]"
        self._add_result(
            f"value_range:{column}",
            passed,
            f"{violations:,} violations",
            f"within {range_str}",
        )

    def check_allowed_values(self, column: str, allowed: list):
        """Check columns only consist of allowed value."""
        violations = self.df.filter(~F.col(column).isin(allowed)).count()
        passed     = violations == 0
        self._add_result(
            f"allowed_values:{column}",
            passed,
            f"{violations:,} invalid values",
            f"must be in {allowed}",
        )

    def check_null_percentage(self, column: str, max_pct: float):
        """NULL tolerance in certain percentage."""
        null_count = self.df.filter(F.col(column).isNull()).count()
        pct        = round(null_count / self.total_rows * 100, 2)
        passed     = pct <= max_pct
        self._add_result(
            f"null_pct:{column}",
            passed,
            f"{pct}% nulls",
            f"<= {max_pct}%",
        )

    def check_freshness(self, date_column: str, max_days_old: int):
        """Check data is not stale (important for a routine pipeline)."""
        max_date = self.df.agg(F.max(date_column)).collect()[0][0]
        if max_date:
            days_old = (datetime.now().date() - max_date).days
            passed   = days_old <= max_days_old
            self._add_result(
                f"freshness:{date_column}",
                passed,
                f"newest record: {max_date} ({days_old} days ago)",
                f"<= {max_days_old} days old",
            )

    def check_referential_integrity(self, column: str, ref_df: DataFrame,
                                    ref_column: str):
        """Checkk foreign key in reference table."""
        orphans = (
            self.df
            .join(ref_df.select(ref_column), self.df[column] == ref_df[ref_column], "left_anti")
            .count()
        )
        passed = orphans == 0
        self._add_result(
            f"ref_integrity:{column}",
            passed,
            f"{orphans:,} orphan records",
            "0 orphans",
        )

    # ── Summary ───────────────────────────────────────────────

    def summary(self) -> dict:
        total  = len(self.results)
        passed = sum(1 for r in self.results if r["passed"])
        failed = total - passed
        score  = round(passed / total * 100, 1) if total else 0

        print(f"\n{'='*50}")
        print(f"  DQ Summary: {self.table_name} ({self.layer})")
        print(f"  Passed : {passed}/{total}  |  Score: {score}%")
        if failed:
            print(f"  ⚠️  {failed} check(s) FAILED — review required")
        print(f"{'='*50}\n")

        if score < 80:
            raise Exception(
                f"Data quality score {score}% is below 80% threshold. "
                "Pipeline halted."
            )
        return {"score": score, "passed": passed, "failed": failed,
                "total": total}

    def save_results(self):
        """Store results to Delta table for historical monitoring."""
        results_df = spark.createDataFrame(self.results)
        (
            results_df.write
            .format("delta")
            .mode("append")
            .saveAsTable("dq_results")
        )
    
        print("DQ results saved to managed table: dq_results")


## Run Checks to The Silver Layer

In [0]:
print("Running Data Quality Checks — Silver Layer\n")

silver_df = spark.table("silver_orders")
dq        = DataQualityChecker(silver_df, "silver_orders", "silver")

# Row count
dq.check_row_count(min_rows=40_000)

# Null checks pada kolom kritikal
dq.check_no_nulls(["order_id", "customer_id", "product_id",
                   "unit_price", "quantity", "status"])

# Primary key unik
dq.check_uniqueness(["order_id"])

# Range nilai wajar
dq.check_value_range("unit_price", min_val=0.01, max_val=10_000)
dq.check_value_range("quantity",   min_val=1,    max_val=100)
dq.check_value_range("total_amount", min_val=0.01)

# Nilai yang diizinkan
dq.check_allowed_values(
    "status",
    ["completed", "cancelled", "returned", "pending"]
)

# Freshness — data tidak boleh lebih dari 400 hari (data 2023)
dq.check_freshness("order_date", max_days_old=400)

# Summary & simpan hasil
silver_summary = dq.summary()
dq.save_results()

Running Data Quality Checks — Silver Layer

  ✅ PASS  row_count: 50,000 rows (threshold: >= 40,000)
  ✅ PASS  no_nulls:order_id: 0 nulls (0.0%) (threshold: 0 nulls)
  ✅ PASS  no_nulls:customer_id: 0 nulls (0.0%) (threshold: 0 nulls)
  ✅ PASS  no_nulls:product_id: 0 nulls (0.0%) (threshold: 0 nulls)
  ✅ PASS  no_nulls:unit_price: 0 nulls (0.0%) (threshold: 0 nulls)
  ✅ PASS  no_nulls:quantity: 0 nulls (0.0%) (threshold: 0 nulls)
  ✅ PASS  no_nulls:status: 0 nulls (0.0%) (threshold: 0 nulls)
  ✅ PASS  uniqueness:order_id: 0 duplicates (threshold: 0 duplicates)
  ✅ PASS  value_range:unit_price: 0 violations (threshold: within [0.01, 10000])
  ✅ PASS  value_range:quantity: 0 violations (threshold: within [1, 100])
  ✅ PASS  value_range:total_amount: 0 violations (threshold: within [0.01, None])
  ✅ PASS  allowed_values:status: 0 invalid values (threshold: must be in ['completed', 'cancelled', 'returned', 'pending'])
  ❌ FAIL  freshness:order_date: newest record: 2023-12-31 (859 days ago) (

## Run Checks to The Gold Layer

In [0]:
print("Running Data Quality Checks — Gold Layer\n")

gold_df = spark.table("monthly_revenue_by_category")
dq_gold = DataQualityChecker(gold_df, "monthly_revenue_by_category", "gold")

dq_gold.check_row_count(min_rows=10)
dq_gold.check_no_nulls(["year_month", "category", "total_revenue"])
dq_gold.check_value_range("total_revenue", min_val=0)
dq_gold.check_value_range("avg_order_value", min_val=0)

gold_summary = dq_gold.summary()
dq_gold.save_results()

Running Data Quality Checks — Gold Layer

  ✅ PASS  row_count: 84 rows (threshold: >= 10)
  ✅ PASS  no_nulls:year_month: 0 nulls (0.0%) (threshold: 0 nulls)
  ✅ PASS  no_nulls:category: 0 nulls (0.0%) (threshold: 0 nulls)
  ✅ PASS  no_nulls:total_revenue: 0 nulls (0.0%) (threshold: 0 nulls)
  ✅ PASS  value_range:total_revenue: 0 violations (threshold: within [0, None])
  ✅ PASS  value_range:avg_order_value: 0 violations (threshold: within [0, None])

  DQ Summary: monthly_revenue_by_category (gold)
  Passed : 6/6  |  Score: 100.0%

DQ results saved to managed table: dq_results


## Part 2: Monitoring Dashboard — Query DQ Results

In [0]:
# Tampilkan semua hasil DQ checks
print("=== Data Quality History ===")
spark.sql("""
    SELECT
        run_time,
        layer,
        table,
        check,
        CASE WHEN passed THEN '✅' ELSE '❌' END AS status,
        value,
        threshold
    FROM dq_results
    ORDER BY run_time DESC, layer, check
""").show(50, truncate=False)

=== Data Quality History ===
+--------------------------+------+---------------------------+---------------------------+------+----------------------------------------+------------------------------------------------------------+
|run_time                  |layer |table                      |check                      |status|value                                   |threshold                                                   |
+--------------------------+------+---------------------------+---------------------------+------+----------------------------------------+------------------------------------------------------------+
|2026-05-08T03:56:21.981595|gold  |monthly_revenue_by_category|no_nulls:category          |✅     |0 nulls (0.0%)                          |0 nulls                                                     |
|2026-05-08T03:56:21.981595|gold  |monthly_revenue_by_category|no_nulls:total_revenue     |✅     |0 nulls (0.0%)                          |0 nulls                     

In [0]:
# Score per run
print("=== DQ Score per Run ===")
spark.sql("""
    SELECT
        run_time,
        table,
        layer,
        COUNT(*) AS total_checks,
        SUM(CASE WHEN passed THEN 1 ELSE 0 END) AS passed,
        ROUND(SUM(CASE WHEN passed THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)
            AS score_pct
    FROM dq_results
    GROUP BY run_time, table, layer
    ORDER BY run_time DESC
""").show(truncate=False)

=== DQ Score per Run ===
+--------------------------+---------------------------+------+------------+------+---------+
|run_time                  |table                      |layer |total_checks|passed|score_pct|
+--------------------------+---------------------------+------+------------+------+---------+
|2026-05-08T03:56:21.981595|monthly_revenue_by_category|gold  |6           |6     |100.0    |
|2026-05-08T03:56:05.434852|silver_orders              |silver|13          |12    |92.3     |
|2026-05-08T03:54:44.873301|silver_orders              |silver|13          |12    |92.3     |
+--------------------------+---------------------------+------+------------+------+---------+



In [0]:
# Failed checks saja — untuk alert/debugging
print("=== Failed Checks ===")
spark.sql("""
    SELECT run_time, layer, table, check, value, threshold
    FROM dq_results
    WHERE NOT passed
    ORDER BY run_time DESC
""").show(truncate=False)

=== Failed Checks ===
+--------------------------+------+-------------+--------------------+----------------------------------------+---------------+
|run_time                  |layer |table        |check               |value                                   |threshold      |
+--------------------------+------+-------------+--------------------+----------------------------------------+---------------+
|2026-05-08T03:56:05.434852|silver|silver_orders|freshness:order_date|newest record: 2023-12-31 (859 days ago)|<= 400 days old|
|2026-05-08T03:54:44.873301|silver|silver_orders|freshness:order_date|newest record: 2023-12-31 (859 days ago)|<= 400 days old|
+--------------------------+------+-------------+--------------------+----------------------------------------+---------------+

